# A1 — ResNet-18 vs DeiT-Tiny trên CIFAR-100

8 cấu hình chính: CNN/Transformer × scratch/head-only/partial/full; thêm2 scratch noaug để kiểm chứng xử lý sau EDA. Đây là notebook mới, độc lập với Stanford Dogs/Pet/Food-101.

**Giả thuyết:** pretraining có thể có lợi trong ngân sách nhỏ; freeze là đánh đổi cần đo; crop/flip trên ảnh32px có thể cải thiện scratch. Không kết luận trước khi có số liệu. Hai process độc lập train trên T4×2; chỉ benchmark tuần tự sau khi cả hai thoát.

Theo `a1_co5085.md`, CIFAR-100 được phép. Nếu đã đăng ký dataset khác tại A1.1, việc đổi dataset cần giảng viên đồng ý. Notebook không thay thế PDF hoặc landing page. Không tự xuất bản website.

Chạy smoke từ kernel sạch trước, sau đó sửa CONFIG sang full và chạy lại toàn bộ. Full mặc định20 epoch/run, cho phép đổi đồng loạt trước study; một seed/ngân sách hữu hạn không chứng minh hội tụ hoặc ý nghĩa thống kê.


## 1. Môi trường và tái lập


### Kiểm tra thư viện; giữ CUDA build của Kaggle

**Đầu vào:** runtime Kaggle có torch/torchvision tương thích. **Đầu ra:** timm1.0.22, các thư viện phân tích và thư mục module tự chứa. Không tự cài lại torch/torchvision.


In [ ]:
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
for name in ('torch','torchvision'):
    if importlib.util.find_spec(name) is None:
        raise RuntimeError('Chọn runtime Kaggle PyTorch có torch/torchvision tương thích trước khi chạy.')
import torch, torchvision
assert hasattr(torch.amp, 'GradScaler'), 'Cần torch>=2.3 và torchvision tương thích'
for name, package in [('numpy','numpy<3'),('pandas','pandas'),('matplotlib','matplotlib'),('sklearn','scikit-learn'),('PIL','Pillow')]:
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable,'-m','pip','install',package])
if importlib.util.find_spec('timm') is None or importlib.metadata.version('timm') != '1.0.22':
    if 'timm' in sys.modules: raise RuntimeError('Cài timm==1.0.22 rồi restart kernel')
    subprocess.check_call([sys.executable,'-m','pip','install','timm==1.0.22'])
BASE = Path('/kaggle/working' if Path('/kaggle/working').exists() else '.').resolve()
RUNTIME = BASE/'a1_cifar100_modules'
RUNTIME.mkdir(parents=True, exist_ok=True)
sys.path.insert(0,str(RUNTIME))
print('torch',torch.__version__,'torchvision',torchvision.__version__,'GPUs',torch.cuda.device_count())


### Protocol, fingerprint và RNG theo process

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile {RUNTIME}/a1_cifar100_common.py
"""Protocol and process-local reproducibility for the standalone CIFAR-100 study."""
import hashlib
import json
import os
import platform
import random
import threading
import time
from pathlib import Path

import numpy as np
import torch
import torchvision
import timm

PROTOCOL = 'cifar100_resnet18_deit_v1'
MODELS = {'resnet18': 'resnet18.a1_in1k', 'deit_tiny': 'deit_tiny_patch16_224.fb_in1k'}
STRATEGIES = ('scratch', 'head_only', 'partial', 'full')
MAIN_IDS = [f'{family}_{strategy}' for family in MODELS for strategy in STRATEGIES]
ALL_IDS = MAIN_IDS + [f'{family}_scratch_noaug' for family in MODELS]


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    tmp.replace(path)


def atomic_save(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, tmp)
    tmp.replace(path)


def versions():
    import sklearn
    return dict(python=platform.python_version(), torch=str(torch.__version__),
                torchvision=str(torchvision.__version__), timm=timm.__version__,
                numpy=np.__version__, sklearn=sklearn.__version__, cuda=torch.version.cuda)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state() if torch.cuda.is_available() else None)


def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if state['cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state(state['cuda'].cpu())


### CONFIG contract và định danh từng thí nghiệm

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_common.py
def default_config():
    base = Path('/kaggle/working' if Path('/kaggle/working').exists() else '.')
    return dict(data_root=str(base / 'data'), output_root=str(base / 'artifacts'),
                run_mode='smoke', execution='dual', epochs=20, seed=42,
                batch_size=32, accumulation=2, num_workers=2, lr=3e-4, weight_decay=.05,
                warmup_epochs=2, grad_clip=1., experiment_ids=[], resume_from={},
                evaluate_only=False, cpu=False, heartbeat_seconds=30)


def validate_config(c):
    if c['run_mode'] not in ('smoke', 'full') or c['execution'] not in ('dual', 'sequential'):
        raise ValueError('Invalid run_mode/execution')
    for key in ('epochs', 'batch_size', 'accumulation'):
        if not isinstance(c[key], int) or isinstance(c[key], bool) or c[key] < 1:
            raise ValueError(f'{key} must be a positive integer')
    if c['num_workers'] < 0 or not 0 < c['lr'] or c['weight_decay'] < 0 or c['heartbeat_seconds'] <= 0:
        raise ValueError('Invalid loader/optimizer/heartbeat settings')
    if c['cpu'] and (c['execution'] != 'sequential' or c['run_mode'] != 'smoke'):
        raise ValueError('CPU is only for explicit sequential smoke')
    ids = c['experiment_ids'] or ALL_IDS
    if len(ids) != len(set(ids)) or set(ids) - set(ALL_IDS):
        raise ValueError('Unknown or repeated experiment IDs')
    return ids


def describe(experiment):
    if experiment not in ALL_IDS:
        raise ValueError(experiment)
    family = next(f for f in MODELS if experiment.startswith(f + '_'))
    strategy = experiment[len(family) + 1:]
    augmented = not strategy.endswith('_noaug')
    return family, strategy.removesuffix('_noaug'), augmented


def run_config(config, manifest, experiment):
    family, strategy, augmented = describe(experiment)
    recipe = dict(epochs=1 if config['run_mode'] == 'smoke' else config['epochs'],
                  batch_size=config['batch_size'], accumulation=config['accumulation'],
                  lr=config['lr'], weight_decay=config['weight_decay'], grad_clip=config['grad_clip'],
                  warmup_epochs=0 if config['run_mode'] == 'smoke' else min(config['warmup_epochs'], config['epochs']),
                  loss='cross_entropy', image_size=224, interpolation='bicubic', normalization='imagenet',
                  dropout=0., drop_path=0., amp=not config['cpu'],
                  num_workers=config['num_workers'], max_train_batches=2 if config['run_mode'] == 'smoke' else None)
    result = dict(protocol=PROTOCOL, experiment_id=experiment, family=family, model=MODELS[family],
                  strategy=strategy, augmented=augmented, augmentation='crop32_pad4_reflect_flip05' if augmented else 'none',
                  seed=config['seed'], run_mode=config['run_mode'], recipe=recipe, study_fingerprint=config['study_fingerprint'],
                  manifest_hash=digest(manifest), versions=versions(),
                  pretrained_sha256=None if strategy == 'scratch' else config['weights'][family]['sha256'])
    result['fingerprint'] = digest(result)
    return result


def check_identity(saved, requested):
    if saved.get('fingerprint') != requested['fingerprint']:
        raise ValueError('Incompatible checkpoint/artifact fingerprint; use its original study/config')


### Heartbeat độc lập với tiến độ GPU

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_common.py
class Heartbeat:
    """A process-local, independent liveness signal, not proof of GPU progress."""
    def __init__(self, path, seconds=30):
        self.path, self.seconds = Path(path), seconds
        self.stop = threading.Event()
        self.progress = {}
    def emit(self):
        write_json(self.path, dict(pid=os.getpid(), utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                                   **self.progress))
    def loop(self):
        while not self.stop.wait(self.seconds):
            self.emit()
    def __enter__(self):
        self.emit()
        self.thread = threading.Thread(target=self.loop, daemon=True)
        self.thread.start()
        return self
    def __exit__(self, *args):
        self.stop.set()
        self.thread.join()
        self.emit()


def run_log(path):
    """Tee each process's output to a per-run log as well as its queue log."""
    import contextlib
    import sys
    @contextlib.contextmanager
    def capture():
        class Tee:
            def __init__(self, original, file): self.original, self.file = original, file
            def write(self, value):
                self.original.write(value); self.file.write(value); self.file.flush()
            def flush(self): self.original.flush(); self.file.flush()
        with open(path, 'a', buffering=1) as handle:
            with contextlib.redirect_stdout(Tee(sys.stdout, handle)), contextlib.redirect_stderr(Tee(sys.stderr, handle)):
                yield
    return capture()


### Cấu hình tập trung

**Đầu vào:** các lựa chọn runtime/ngân sách. **Đầu ra:** CONFIG. Không sửa CONFIG sau khi freeze study; đổi recipe/epoch tạo fingerprint mới.


In [ ]:
import json
from a1_cifar100_common import default_config, validate_config, ALL_IDS, MAIN_IDS
CONFIG = default_config()
CONFIG['run_mode'] = os.getenv('CO5085_RUN_MODE','smoke')
CONFIG['epochs'] = 20                      # Cùng ngân sách cho cả10 run full
CONFIG['execution'] = 'dual'               # Hai process, GPU0/GPU1
CONFIG['experiment_ids'] = []              # Rỗng = cả10 run
# CONFIG['execution'] = 'sequential'       # Chủ động chọn khi chỉ có1 GPU
# CONFIG.update(cpu=True, execution='sequential')  # CPU chỉ smoke
# CONFIG['data_root'] = '/kaggle/working/data'
# CONFIG['output_root'] = '/kaggle/working/artifacts'
# CONFIG['resume_from'] = {'resnet18_scratch':'last'}
# CONFIG['evaluate_only'] = True           # Chỉ khi đã có đủ checkpoint training
print(json.dumps(CONFIG,indent=2))
print('Selected:',validate_config(CONFIG))


## 2. Dataset và EDA có mục tiêu

Nguồn: [CIFAR-100 chính thức](https://www.cs.toronto.edu/~kriz/cifar.html), Alex Krizhevsky, *Learning Multiple Layers of Features from Tiny Images*,2009. Trang nguồn không nêu rõ giấy phép chuẩn cho toàn bộ ảnh: ghi tình trạng này thay vì tự gán CC/MIT; trích nguồn và kiểm tra điều khoản khi tái phân phối. Dataset được đề A1 liệt kê trực tiếp.

Official train50.000 →45.000 train/5.000 validation bằng stratification seed42; official test10.000 giữ nguyên.100 fine classes,20 coarse superclasses chỉ để EDA, không thay target. Mỗi lớp450/50/100 ảnh.

EDA dùng train để quyết định: phân bố lớp; định dạng/miền pixel; RGB/độ sáng; mẫu mọi lớp và nhóm superclass; ảnh32px so với upscale224px. Validation/test chỉ kiểm tra count/integrity/duplicate. Hash exact pixels không phát hiện mọi near-duplicate; giữ split, công bố cảnh báo và không tuyên bố đã dedup.


### Transform, seed theo ảnh/epoch và DataLoader riêng

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile {RUNTIME}/a1_cifar100_data.py
"""CIFAR-100 split, train-only EDA, and reproducible low-resolution augmentation."""
import hashlib
import json
import pickle
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from a1_cifar100_common import digest, write_json, versions, PROTOCOL, MODELS, file_hash, validate_config

MEAN, STD = (.485, .456, .406), (.229, .224, .225)


def pipeline(augmented):
    steps = [transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
             transforms.RandomHorizontalFlip(.5)] if augmented else []
    return transforms.Compose(steps + [transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
                                      transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


class IndexedCIFAR(Dataset):
    def __init__(self, raw, indices, augmented, seed, epoch):
        self.raw, self.indices = raw, list(indices)
        self.transform, self.seed, self.epoch = pipeline(augmented), seed, epoch
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item])
        # Stateless augmentation: independent of model RNG, worker count, scheduling.
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed(self.seed + self.epoch * 1000003 + index)
            image = self.transform(Image.fromarray(self.raw.data[index]))
        return image, int(self.raw.targets[index]), index


def seed_worker(worker_id):
    import random
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)


def make_loader(raw, indices, config, *, train=False, augmented=False, epoch=0):
    generator = torch.Generator().manual_seed(config['seed'] + 1000003 * epoch + (0 if train else 99991))
    dataset = IndexedCIFAR(raw, indices, augmented, config['seed'], epoch)
    return DataLoader(dataset, batch_size=config['batch_size'], shuffle=train,
                      num_workers=config['num_workers'], generator=generator, worker_init_fn=seed_worker,
                      persistent_workers=False, pin_memory=not config['cpu'],
                      **({'multiprocessing_context':'spawn'} if config['num_workers'] else {}))


def load_raw(config, download=False):
    train = datasets.CIFAR100(config['data_root'], train=True, download=download)
    test = datasets.CIFAR100(config['data_root'], train=False, download=download)
    return train, test


### Split kiểm chứng và audit exact duplicates

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_data.py
def build_manifest(train, test, seed):
    for raw, size in ((train, 50000), (test, 10000)):
        if raw.data.shape != (size, 32, 32, 3) or raw.data.dtype != np.uint8:
            raise ValueError('CIFAR-100 shape/dtype mismatch')
        if not np.array_equal(np.bincount(raw.targets, minlength=100), np.full(100, size // 100)):
            raise ValueError('Invalid labels/counts; expected all100 fine categories')
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=5000, random_state=seed)
    tr, va = next(splitter.split(np.zeros(50000), train.targets))
    assert not set(tr) & set(va)
    parts = dict(train=tr.tolist(), val=va.tolist(), test=list(range(10000)))
    hashes, groups = {}, defaultdict(list)
    for split, raw in (('train', train), ('test', test)):
        h = hashlib.sha256(raw.data.tobytes())
        h.update(np.asarray(raw.targets, dtype=np.int64).tobytes())
        hashes[split] = h.hexdigest()
    # Metadata names/IDs and raw pixels all contribute to identity.
    return dict(dataset='CIFAR-100', seed=seed, indices=parts, classes=list(train.classes),
                raw_hashes=hashes, sizes={k: len(v) for k, v in parts.items()})


def duplicate_audit(train, test, manifest):
    groups = defaultdict(list)
    for split, indices in manifest['indices'].items():
        raw = test if split == 'test' else train
        for i in indices:
            groups[hashlib.sha256(raw.data[i].tobytes()).hexdigest()].append(
                dict(split=split, index=i, label=int(raw.targets[i])))
    repeated = [items for items in groups.values() if len(items) > 1]
    return dict(groups=repeated, duplicate_groups=len(repeated),
                cross_split_groups=sum(len({x['split'] for x in items}) > 1 for items in repeated),
                conflicting_label_groups=sum(len({x['label'] for x in items}) > 1 for items in repeated),
                policy='Preserve registered split; exact pixels only, near-duplicates not audited')


### EDA: phân bố, RGB, sample grid, superclass và upscale

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_data.py
def eda(train, test, manifest, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    indices = manifest['indices']
    counts = pd.DataFrame({'class': manifest['classes'], **{
        split: np.bincount(np.asarray((test if split == 'test' else train).targets)[ids], minlength=100)
        for split, ids in indices.items()}})
    counts.to_csv(output / 'eda_counts.csv', index=False)
    counts.set_index('class').plot.bar(figsize=(22, 5), width=.9)
    plt.ylabel('Images'); plt.tight_layout(); plt.savefig(output / 'eda_distribution.png', dpi=140); plt.close()
    # Only train pixels influence descriptive statistics and preprocessing choices.
    x = train.data[indices['train']]
    sums, squares, n = np.zeros(3), np.zeros(3), 0
    for start in range(0, len(x), 512):
        chunk = x[start:start+512].astype(np.float64) / 255
        sums += chunk.sum(axis=(0, 1, 2)); squares += (chunk * chunk).sum(axis=(0, 1, 2))
        n += np.prod(chunk.shape[:3])
    mean = sums / n
    stats = dict(train_rgb_mean=mean.tolist(), train_rgb_std=np.sqrt(np.maximum(0, squares/n - mean**2)).tolist(),
                 shape=list(train.data.shape[1:]), dtype=str(x.dtype), pixel_range=[int(x.min()), int(x.max())],
                 class_count_min=int(counts.train.min()), class_count_max=int(counts.train.max()),
                 test_use='Integrity/counts/duplicates only; no test EDA used to choose transforms')
    write_json(output / 'eda_statistics.json', stats)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for channel, color in enumerate(('red', 'green', 'blue')):
        hist = np.bincount(x[..., channel].ravel(), minlength=256)
        axes[0].plot(np.arange(256), hist / hist.sum(), color=color)
    axes[0].set(title='Train RGB histogram', xlabel='Pixel value', ylabel='Frequency')
    axes[1].hist(x[::10].mean(axis=(1, 2, 3)), bins=40)
    axes[1].set(title='Train mean intensity (every10th train sample)', xlabel='Mean RGB intensity')
    fig.tight_layout(); fig.savefig(output / 'eda_color.png', dpi=140); plt.close(fig)
    labels = np.asarray(train.targets)
    fig, axes = plt.subplots(10, 10, figsize=(17, 17))
    examples = {}
    for category, axis in enumerate(axes.flat):
        ids = [i for i in indices['train'] if labels[i] == category]
        examples[category] = ids[:4]
        axis.imshow(train.data[ids[0]]); axis.set_title(manifest['classes'][category], fontsize=7); axis.axis('off')
    fig.tight_layout(); fig.savefig(output / 'eda_all_classes.png', dpi=140); plt.close(fig)
    # Official coarse labels are used only for EDA, never as training targets.
    base = Path(train.root) / train.base_folder
    with (base / 'meta').open('rb') as handle:
        coarse_names = pickle.load(handle, encoding='latin1')['coarse_label_names']
    with (base / 'train').open('rb') as handle:
        coarse = np.asarray(pickle.load(handle, encoding='latin1')['coarse_labels'])
    chosen = []
    for super_id in (0, 1):
        cats = sorted(set(labels[np.asarray(indices['train'])[coarse[indices['train']] == super_id]]))[:3]
        chosen += [(cat, coarse_names[super_id]) for cat in cats]
    fig, axes = plt.subplots(len(chosen), 4, figsize=(10, 2 * len(chosen)))
    for row, (category, superclass) in enumerate(chosen):
        for axis, index in zip(axes[row], examples[category]):
            axis.imshow(train.data[index]); axis.set_title(f'{superclass}\n{manifest["classes"][category]}', fontsize=8); axis.axis('off')
    fig.tight_layout(); fig.savefig(output / 'eda_superclasses.png', dpi=140); plt.close(fig)
    index = indices['train'][0]
    original = Image.fromarray(train.data[index])
    fig, axes = plt.subplots(1, 5, figsize=(14, 3))
    axes[0].imshow(original, interpolation='nearest'); axes[0].set_title('32x32 original')
    axes[1].imshow(original.resize((224, 224), Image.Resampling.BICUBIC)); axes[1].set_title('224x224 bicubic')
    for epoch, axis in enumerate(axes[2:]):
        tensor, _, _ = IndexedCIFAR(train, [index], True, manifest['seed'], epoch)[0]
        axis.imshow((tensor * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1).permute(1, 2, 0))
        axis.set_title(f'Crop32 + flip, view{epoch+1}')
    for axis in axes: axis.axis('off')
    fig.tight_layout(); fig.savefig(output / 'eda_preprocessing.png', dpi=150); plt.close(fig)
    audit = duplicate_audit(train, test, manifest)
    write_json(output / 'duplicate_audit.json', audit)
    balanced = stats['class_count_min'] == stats['class_count_max']
    decisions = (f'Train class counts min/max: {stats["class_count_min"]}/{stats["class_count_max"]}. '
                 f'Balanced={balanced}; no weighting/sampler correction.\n'
                 '32px images: keep low-resolution content; crop32/reflection-pad4/flip before resize224.\n'
                 'Hypothesis only: compare scratch augmentation vs noaug for each family, same budget.\n'
                 'ImageNet normalization is fixed to match pretrained inputs, not estimated using test.\n'
                 f'Exact cross-split duplicate groups={audit["cross_split_groups"]}; no automatic deduplication.\n')
    (output / 'eda_decisions.md').write_text(decisions)
    return stats


### Chuẩn bị dữ liệu, cache weights và study manifest

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_data.py
def cache_weights(config):
    import timm
    import torch
    from a1_cifar100_common import atomic_save
    folder = Path(config['data_root']) / 'cifar100_weights_timm1022'
    folder.mkdir(parents=True, exist_ok=True)
    weights = {}
    for family, model_id in MODELS.items():
        path = folder / f'{family}.pt'
        provenance = path.with_suffix('.json')
        if path.exists():
            info = json.loads(provenance.read_text())
            if info['model'] != model_id or info['timm'] != timm.__version__ or info['sha256'] != file_hash(path):
                raise ValueError(f'Invalid cached pretrained checkpoint: {path}')
        else:
            # CPU only: no model retained in parent GPU memory.
            model = timm.create_model(model_id, pretrained=True, num_classes=1000)
            atomic_save(path, model.state_dict())
            write_json(provenance, dict(model=model_id, timm=timm.__version__, sha256=file_hash(path),
                                        source=dict(model.pretrained_cfg)))
            del model
        weights[family] = dict(path=str(path.resolve()), sha256=file_hash(path))
    return weights


def prepare_data(config):
    validate_config(config)
    train, test = load_raw(config, download=True)
    manifest = build_manifest(train, test, config['seed'])
    root = Path(config['output_root']).resolve() / 'A1' / PROTOCOL / 'preparation' / digest(manifest)
    root.mkdir(parents=True, exist_ok=True)
    write_json(root / 'manifest.json', manifest)
    eda(train, test, manifest, root)
    return manifest, root


def prepare(config, manifest, eda_root):
    validate_config(config)
    if versions()['timm'] != '1.0.22':
        raise RuntimeError('This protocol requires timm==1.0.22')
    config = dict(config)
    config['weights'] = cache_weights(config)
    sources = {p.name: file_hash(p) for p in Path(__file__).parent.glob('a1_cifar100_*.py')}
    identity = dict(protocol=PROTOCOL, manifest_hash=digest(manifest), versions=versions(), sources=sources,
                    recipe={k: v for k, v in config.items() if k not in (
                        'data_root', 'output_root', 'experiment_ids', 'resume_from', 'evaluate_only', 'heartbeat_seconds', 'weights', 'root', 'study_fingerprint')},
                    weights={f: w['sha256'] for f, w in config['weights'].items()})
    study = digest(identity)
    root = Path(config['output_root']).resolve() / 'A1' / PROTOCOL / study / config['run_mode']
    root.mkdir(parents=True, exist_ok=True)
    config['study_fingerprint'], config['root'] = study, str(root)
    write_json(root / 'study_identity.json', identity)
    write_json(root / 'manifest.json', manifest)
    np.savez(root / 'split.npz', **{k: np.asarray(v) for k, v in manifest['indices'].items()})
    # Resume/evaluate operational settings are intentionally excluded from study identity.
    write_json(root / 'launch.json', config)
    import shutil
    for path in Path(eda_root).iterdir():
        if path.is_file(): shutil.copy2(path, root/path.name)
    print('Study:', root, '\nSizes:', manifest['sizes'])
    return config, manifest


### Chạy EDA trên dữ liệu thật

**Đầu vào:** CIFAR-100, tải lần đầu nếu cần. **Đầu ra:** manifest và các hình/bảng EDA. Thiếu/corrupt/download lỗi dừng ngay; không thay bằng ảnh giả.


In [ ]:
from a1_cifar100_data import prepare_data
manifest, EDA_ROOT = prepare_data(CONFIG)
from IPython.display import display, Image as DisplayImage, Markdown
import pandas as pd
counts = pd.read_csv(EDA_ROOT/'eda_counts.csv')
display(counts)
display(pd.DataFrame([json.loads((EDA_ROOT/'eda_statistics.json').read_text())]))
for filename in ('eda_distribution.png','eda_color.png','eda_all_classes.png','eda_superclasses.png','eda_preprocessing.png'):
    display(DisplayImage(filename=str(EDA_ROOT/filename)))
display(Markdown((EDA_ROOT/'eda_decisions.md').read_text()))
audit = json.loads((EDA_ROOT/'duplicate_audit.json').read_text())
print({key:value for key,value in audit.items() if key != 'groups'})


## 3. Từ EDA đến quyết định xử lý và đối chứng

- Nếu count xác nhận cân bằng450 ảnh/lớp, không dùng class weighting/weighted sampler.
- Ảnh32px có ít chi tiết: upscale224 chỉ để tương thích cả hai kiến trúc, không tạo thêm thông tin. Giữ ResNet stem và DeiT patch16 tiêu chuẩn để so pretrained/scratch cùng kiến trúc.
- **Hướng xử lý được đo:** crop32 pad4 phản chiếu + flip0.5 trước resize224 bicubic. Tránh crop mạnh ảnh đã upscale. ImageNet normalization dùng chung để tương thích checkpoint, không học từ test.
- Baseline/noaug và validation/test: resize224 bicubic →tensor →ImageNet normalize, không crop/flip ngẫu nhiên.
-8 run chính dùng crop/flip. Hai run `*_scratch_noaug` chỉ thay augmentation, giữ split, initialization seed, AdamW/LR/weight decay/budget. Delta validation/test/F1/generalization gap mới quyết định lợi/hại; không mặc định augmentation luôn tốt.
- Train augmented và train-eval là hai phép đo khác nhau: generalization gap dùng FP32 train-eval không augmentation trừ FP32 validation accuracy, cùng best checkpoint.


## 4. Training dùng chung, chia thành từng bước


### Freeze invariants và phục hồi weights/buffers/RNG

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile {RUNTIME}/a1_cifar100_train.py
"""Training primitives: actual-image accumulation, FP32 selection, epoch resume."""
import copy
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from a1_cifar100_common import (atomic_save, write_json, check_identity, rng_state,
                                restore_rng, run_config, file_hash)
from a1_cifar100_data import make_loader


def frozen_eval(model):
    for module in model.modules():
        params = list(module.parameters())
        if params and not any(p.requires_grad for p in params):
            module.eval()


def freeze_probe(model, batch, device, strategy):
    state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
    rng, modes = rng_state(), {name: m.training for name, m in model.named_modules()}
    frozen = {name for name, p in model.named_parameters() if not p.requires_grad}
    for name, module in model.named_modules():
        params = list(module.parameters())
        if params and not any(p.requires_grad for p in params):
            frozen.update(f'{name}.{key}' if name else key for key, _ in module.named_buffers(recurse=False))
    try:
        model.train(); frozen_eval(model)
        x, y = batch[0][:2].to(device), batch[1][:2].to(device)
        optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=.01)
        logits = model(x)
        if logits.shape != (len(x), 100):
            raise ValueError(f'Unexpected logits {logits.shape}')
        nn.functional.cross_entropy(logits, y).backward(); optimizer.step()
        for name in frozen:
            torch.testing.assert_close(model.state_dict()[name].cpu(), state[name], rtol=0, atol=0)
    finally:
        model.load_state_dict(state)
        model.zero_grad(set_to_none=True)
        restore_rng(rng)
        for name, module in model.named_modules():
            module.training = modes[name]


### Gradient accumulation chuẩn hóa theo số ảnh thật

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_train.py
def optimizer_step(model, optimizer, scaler, item_count, clip):
    # The gradients accumulated from reduction='sum' become the true mean over
    # this optimizer group, even with unequal microbatches or a single tail batch.
    scaler.unscale_(optimizer)
    for p in model.parameters():
        if p.grad is not None:
            p.grad.div_(item_count)
    if clip is not None:
        nn.utils.clip_grad_norm_(model.parameters(), clip)
    # GradScaler may skip optimizer.step() on non-finite gradients. Its return
    # value cannot identify this: a successful AdamW.step() also returns None.
    updated = False
    def record_update(optimizer, args, kwargs):
        nonlocal updated
        updated = True
    with optimizer.register_step_post_hook(record_update):
        scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    return updated


def train_epoch(model, loader, optimizer, scaler, device, accumulation, clip, max_batches=None):
    model.train(); frozen_eval(model)
    optimizer.zero_grad(set_to_none=True)
    items = correct = group_items = group_batches = 0
    optimizer_steps = step_attempts = 0
    loss_sum = 0.
    if device.type == 'cuda': torch.cuda.synchronize(device)
    start = time.perf_counter()
    for batch_id, (images, labels, _) in enumerate(loader):
        if max_batches is not None and batch_id >= max_batches: break
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(images)
            loss = nn.functional.cross_entropy(logits, labels, reduction='sum')
        scaler.scale(loss).backward()
        n = len(labels)
        items += n; group_items += n; group_batches += 1
        loss_sum += float(loss.detach())
        correct += int((logits.argmax(1) == labels).sum())
        if group_batches == accumulation:
            optimizer_steps += optimizer_step(model, optimizer, scaler, group_items, clip)
            step_attempts += 1
            group_items = group_batches = 0
    if group_items:
        optimizer_steps += optimizer_step(model, optimizer, scaler, group_items, clip)
        step_attempts += 1
    if device.type == 'cuda': torch.cuda.synchronize(device)
    if items == 0: raise ValueError('Empty training loader')
    return dict(loss=loss_sum/items, accuracy=correct/items, items=items,
                optimizer_steps=optimizer_steps, skipped_steps=step_attempts-optimizer_steps,
                seconds=time.perf_counter()-start)


### Evaluation FP32 và scheduler warmup/cosine

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_train.py
@torch.no_grad()
def evaluate(model, loader, device, max_batches=None):
    model.eval()
    ys, ps, ids, probs = [], [], [], []
    total = 0.
    # No autocast: checkpoint selection and all final metrics use FP32.
    for batch_id, (images, labels, indices) in enumerate(loader):
        if max_batches is not None and batch_id >= max_batches: break
        logits = model(images.to(device).float())
        total += float(nn.functional.cross_entropy(logits, labels.to(device), reduction='sum'))
        probabilities = logits.softmax(1).cpu().numpy()
        ys.extend(labels.tolist()); ps.extend(probabilities.argmax(1).tolist())
        ids.extend(indices.tolist()); probs.append(probabilities)
    if not ys: raise ValueError('Empty evaluation loader')
    return dict(loss=total/len(ys), accuracy=float(np.mean(np.asarray(ys) == ps)), items=len(ys)), dict(
        y_true=np.asarray(ys), y_pred=np.asarray(ps), indices=np.asarray(ids), probabilities=np.concatenate(probs))


def scheduler_for(optimizer, epochs, warmup):
    def factor(epoch):
        if epoch < warmup: return (epoch + 1) / warmup
        # Epoch0 is full LR without warmup; final trained epoch remains >0.
        progress = (epoch - warmup) / max(1, epochs - warmup)
        return .5 * (1 + math.cos(math.pi * min(1, progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


### Train loop, chọn best bằng validation và checkpoint epoch

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_train.py
def train_run(model, raw_train, config, manifest, experiment, device, heartbeat):
    rc = run_config(config, manifest, experiment)
    folder = Path(config['root']) / experiment
    folder.mkdir(parents=True, exist_ok=True)
    cfg_path = folder / 'config.json'
    if cfg_path.exists(): check_identity(json.loads(cfg_path.read_text()), rc)
    recipe = rc['recipe']
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=recipe['lr'], weight_decay=recipe['weight_decay'])
    scheduler = scheduler_for(optimizer, recipe['epochs'], recipe['warmup_epochs'])
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    history, next_epoch, best_acc, best_loss, best_epoch, best_model = [], 0, -1., float('inf'), -1, None
    resume = config['resume_from'].get(experiment)
    if resume:
        path = folder / 'last.pt' if resume == 'last' else Path(resume)
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
        check_identity(checkpoint['config'], rc)  # Before any artifact mutation.
        model.load_state_dict(checkpoint['model']); optimizer.load_state_dict(checkpoint['optimizer'])
        scheduler.load_state_dict(checkpoint['scheduler']); scaler.load_state_dict(checkpoint['scaler'])
        history, next_epoch = checkpoint['history'], checkpoint['next_epoch']
        if [r['epoch'] for r in history] != list(range(next_epoch)):
            raise ValueError('Checkpoint history is not contiguous')
        best_acc, best_loss, best_epoch, best_model = [checkpoint[k] for k in ('best_acc', 'best_loss', 'best_epoch', 'best_model')]
        restore_rng(checkpoint['rng'])
    write_json(cfg_path, rc)
    write_json(folder / 'environment.json', dict(versions=rc['versions'], device=str(device),
               device_name=torch.cuda.get_device_name(device) if device.type == 'cuda' else 'cpu',
               training_precision='amp_fp16' if device.type == 'cuda' else 'fp32',
               execution=config['execution'], effective_batch_target=recipe['batch_size']*recipe['accumulation']))
    train_ids, val_ids = manifest['indices']['train'], manifest['indices']['val']
    if config['run_mode'] == 'smoke':
        # Real samples, small batch budget; sparse coverage is explicitly non-reportable.
        train_ids, val_ids = train_ids[:max(2*config['batch_size'], 2)], val_ids[:config['batch_size']]
    probe_loader = make_loader(raw_train, train_ids, config, train=True, augmented=rc['augmented'])
    freeze_probe(model, next(iter(probe_loader)), device, rc['strategy'])
    for epoch in range(next_epoch, recipe['epochs']):
        heartbeat.progress = dict(experiment=experiment, phase='train', epoch=epoch+1)
        if device.type == 'cuda': torch.cuda.reset_peak_memory_stats(device)
        used_lr = optimizer.param_groups[0]['lr']
        train = train_epoch(model, make_loader(raw_train, train_ids, config, train=True,
                            augmented=rc['augmented'], epoch=epoch), optimizer, scaler, device,
                            recipe['accumulation'], recipe['grad_clip'], recipe['max_train_batches'])
        peak = torch.cuda.max_memory_allocated(device)/2**20 if device.type == 'cuda' else 0.
        heartbeat.progress = dict(experiment=experiment, phase='validation', epoch=epoch+1)
        val, _ = evaluate(model, make_loader(raw_train, val_ids, config), device)
        history.append(dict(epoch=epoch, train_loss=train['loss'], train_accuracy=train['accuracy'],
                            val_loss=val['loss'], val_accuracy=val['accuracy'], training_seconds=train['seconds'],
                            peak_training_memory_mb=peak, learning_rate=used_lr, train_items=train['items'],
                            optimizer_steps=train['optimizer_steps'], skipped_steps=train['skipped_steps']))
        improved = val['accuracy'] > best_acc or (val['accuracy'] == best_acc and val['loss'] < best_loss)
        if improved:
            best_acc, best_loss, best_epoch = val['accuracy'], val['loss'], epoch
            best_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if train['optimizer_steps'] > 0:
            scheduler.step()
        else:
            print(f'{experiment} epoch={epoch+1}: no optimizer updates; '
                  f'scheduler unchanged (AMP skipped {train["skipped_steps"]} steps)', flush=True)
        state = dict(config=rc, model=model.state_dict(), optimizer=optimizer.state_dict(),
                     scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), rng=rng_state(),
                     history=history, next_epoch=epoch+1, best_acc=best_acc, best_loss=best_loss,
                     best_epoch=best_epoch, best_model=best_model)
        atomic_save(folder / 'last.pt', state)
        if improved:
            atomic_save(folder / 'best.pt', dict(config=rc, model=best_model, epoch=best_epoch))
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        print(f'{experiment} epoch={epoch+1}/{recipe["epochs"]} train={train["accuracy"]:.4f} '
              f'val={val["accuracy"]:.4f} updates={train["optimizer_steps"]} '
              f'skipped={train["skipped_steps"]}', flush=True)
    # An external last checkpoint can reconstruct the historical best locally.
    if best_model is None: raise ValueError('No trained best model available')
    atomic_save(folder / 'best.pt', dict(config=rc, model=best_model, epoch=best_epoch))
    pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
    write_json(folder / 'trained.json', dict(fingerprint=rc['fingerprint'], epochs_completed=len(history),
                                           checkpoint_sha256=file_hash(folder / 'best.pt')))


AdamW dùng chung LR3e-4, weight decay0.05; CE, clip1, dropout/drop-path0, không mixup/CutMix/smoothing. Full warmup2/cosine; số epoch cấu hình chung. Batch32×accum2 là mục tiêu64 ảnh/update; nhóm cuối chia theo số ảnh thật. Với BatchNorm, accumulation không tương đương hoàn toàn batch64 thực.

AMP chỉ khi train CUDA; validation/checkpoint selection và final metrics FP32. Checkpoint chọn bằng validation accuracy, hòa bằng validation loss. Last lưu cả best model để resume từ thư mục khác vẫn khôi phục được best. Dừng giữa epoch sẽ chạy lại epoch chưa checkpoint, không resume giữa batch. Scheduler chỉ tăng khi epoch có ít nhất1 optimizer step thực sự; AMP có thể bỏ qua step khi gradient không hữu hạn. Log/history ghi optimizer_steps và skipped_steps; epoch không có update giữ nguyên LR. Không tắt cảnh báo scheduler để che lỗi.


## 5. Hai kiến trúc được định nghĩa tách biệt


**ResNet-18:** convolution backbone, pooled feature512 chiều, `fc`100 lớp. Scratch xây toàn bộ từ khởi tạo mới; ID có `in1k` không tự tải weights. Pretrained nạp ImageNet1000 lớp và bỏ FC không tương thích. Head-only chỉ mở FC; partial mở layer4+FC; full mở toàn bộ. Frozen BatchNorm giữ eval.


### ResNet-18 — khởi tạo và freeze mask riêng

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile {RUNTIME}/a1_cifar100_resnet.py
"""ResNet-18: a separate, readable architecture cell in the notebook."""
import timm
from a1_cifar100_common import seed_all


def build_resnet18(strategy, seed, pretrained_path=None):
    seed_all(seed)
    # False means fresh initialization, even though the ID contains 'in1k'.
    model = timm.create_model('resnet18.a1_in1k', pretrained=False,
                              num_classes=100, drop_rate=0., drop_path_rate=0.)
    if strategy != 'scratch':
        if pretrained_path is None:
            raise ValueError('Pretrained runs require an audited ImageNet checkpoint')
        # timm discards the incompatible 1000-class FC; our 100-class FC stays fresh.
        timm.models.load_pretrained(model, pretrained_cfg={**model.pretrained_cfg, 'file': pretrained_path},
                                    num_classes=100)
    for name, parameter in model.named_parameters():
        parameter.requires_grad = (strategy in ('scratch', 'full') or name.startswith('fc.')
                                   or (strategy == 'partial' and name.startswith('layer4.')))
    return model


**DeiT-Tiny:** patch16 biến input224 thành14×14 patches, embedding192 chiều,12 Transformer blocks; head100 lớp. Scratch không nạp pretrained. Head-only chỉ mở head; partial mở blocks10–11, norm, head; full mở toàn bộ. Dùng kiến trúc của timm vẫn là train from-scratch khi không nạp weights.


### DeiT-Tiny — khởi tạo và freeze mask riêng

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile {RUNTIME}/a1_cifar100_deit.py
"""DeiT-Tiny: distinct from the ResNet architecture cell."""
import timm
from a1_cifar100_common import seed_all


def build_deit_tiny(strategy, seed, pretrained_path=None):
    seed_all(seed)
    # Patch16 at 224px produces 14x14 patches; embed dim192, 12 transformer blocks.
    model = timm.create_model('deit_tiny_patch16_224.fb_in1k', pretrained=False,
                              num_classes=100, drop_rate=0., drop_path_rate=0.)
    if strategy != 'scratch':
        if pretrained_path is None:
            raise ValueError('Pretrained runs require an audited ImageNet checkpoint')
        timm.models.load_pretrained(model, pretrained_cfg={**model.pretrained_cfg, 'file': pretrained_path},
                                    num_classes=100)
    for name, parameter in model.named_parameters():
        parameter.requires_grad = (strategy in ('scratch', 'full') or name.startswith('head.')
                                   or (strategy == 'partial' and name.startswith(('blocks.10.', 'blocks.11.', 'norm.'))))
    return model


## 6. Worker và benchmark có hàng rào process


### Worker CLI độc lập — chỉ train, không benchmark

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile {RUNTIME}/a1_cifar100_worker.py
"""Independent queue process, never benchmarks alongside training."""
import argparse
import json
import os
import traceback
from pathlib import Path

# Must precede CUDA initialization for deterministic CUDA matmul.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
from a1_cifar100_common import (validate_config, describe, run_config, check_identity, Heartbeat,
                                write_json, file_hash, MODELS, run_log)
from a1_cifar100_data import load_raw
from a1_cifar100_train import train_run
from a1_cifar100_resnet import build_resnet18
from a1_cifar100_deit import build_deit_tiny


def build_model(rc, config, device):
    weight = None if rc['strategy'] == 'scratch' else config['weights'][rc['family']]['path']
    if weight and file_hash(weight) != rc['pretrained_sha256']:
        raise ValueError('Pretrained checkpoint bytes changed')
    builder = build_resnet18 if rc['family'] == 'resnet18' else build_deit_tiny
    return builder(rc['strategy'], rc['seed'], weight).to(device)


def trained_valid(folder, rc):
    folder = Path(folder)
    try:
        check_identity(json.loads((folder/'config.json').read_text()), rc)
        trained = json.loads((folder/'trained.json').read_text())
        import pandas as pd
        rows = pd.read_csv(folder/'history.csv')
        return (trained['fingerprint'] == rc['fingerprint'] and trained['epochs_completed'] == rc['recipe']['epochs']
                and rows.epoch.tolist() == list(range(rc['recipe']['epochs']))
                and trained['checkpoint_sha256'] == file_hash(folder/'best.pt'))
    except (OSError, ValueError, KeyError):
        return False


def queue(config_path, family, device_name):
    config = json.loads(Path(config_path).read_text())
    manifest = json.loads((Path(config['root'])/'manifest.json').read_text())
    selected = validate_config(config)
    device = torch.device(device_name)
    if device.type == 'cuda': torch.cuda.set_device(device)
    torch.set_num_threads(2)
    train, _ = load_raw(config)
    failures = []
    for experiment in selected:
        if describe(experiment)[0] != family: continue
        folder = Path(config['root'])/experiment
        folder.mkdir(parents=True, exist_ok=True)
        rc = run_config(config, manifest, experiment)
        # Reject identity changes without altering an existing run.
        if (folder/'config.json').exists():
            check_identity(json.loads((folder/'config.json').read_text()), rc)
        with Heartbeat(folder/'heartbeat.json', config['heartbeat_seconds']) as heartbeat, run_log(folder/'train.log'):
            try:
                if trained_valid(folder, rc):
                    print(f'{experiment}: compatible training already complete', flush=True)
                    continue
                if config['evaluate_only']:
                    raise ValueError(f'{experiment}: evaluate_only requires completed training')
                write_json(folder/'status.json', dict(state='training', fingerprint=rc['fingerprint']))
                model = build_model(rc, config, device)
                try:
                    train_run(model, train, config, manifest, experiment, device, heartbeat)
                finally:
                    del model
                    if device.type == 'cuda': torch.cuda.empty_cache()
                write_json(folder/'status.json', dict(state='trained', fingerprint=rc['fingerprint']))
            except KeyboardInterrupt:
                write_json(folder/'status.json', dict(state='interrupted', fingerprint=rc['fingerprint']))
                raise
            except Exception as exc:
                failures.append(experiment)
                write_json(folder/'status.json', dict(state='failed', error=str(exc), fingerprint=rc['fingerprint']))
                traceback.print_exc()
    if failures: raise RuntimeError(f'Failed experiments: {failures}')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('config'); parser.add_argument('family', choices=list(MODELS)); parser.add_argument('device')
    args = parser.parse_args()
    queue(args.config, args.family, args.device)


### Điều phối process và chờ queue; không chia sẻ RNG

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile {RUNTIME}/a1_cifar100_launch.py
"""Notebook orchestration: distinct nonblocking launch cells, then a join barrier."""
import os
import subprocess
import sys
import time
from pathlib import Path

import torch
from a1_cifar100_common import validate_config, describe


def check_devices(config):
    validate_config(config)
    if config['cpu']: return
    count = torch.cuda.device_count()
    if config['execution'] == 'dual' and count < 2:
        raise RuntimeError('Dual execution needs >=2 GPUs; explicitly set execution="sequential" for one GPU')
    if count < 1:
        raise RuntimeError('CUDA GPU required; explicit cpu=True is allowed only for sequential smoke')


def launch_family(config, family, handles):
    check_devices(config)
    if family in handles: raise RuntimeError(f'{family} already launched; join before re-running cells')
    selected = validate_config(config)
    if not any(describe(experiment)[0] == family for experiment in selected):
        handles[family] = dict(process=None, returncode=0, offset=0)
        return
    if config['execution'] == 'sequential':
        wait_queues(handles)  # An explicit one-GPU fallback never overlaps queues.
    device = 'cpu' if config['cpu'] else ('cuda:1' if family == 'deit_tiny' and config['execution']=='dual' else 'cuda:0')
    root = Path(config['root'])
    log_path = root/f'{family}_queue.log'
    env = {**os.environ, 'PYTHONUNBUFFERED':'1', 'CUBLAS_WORKSPACE_CONFIG':':4096:8',
           'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2'}
    log = log_path.open('w')
    try:
        process = subprocess.Popen([sys.executable, '-u', str(Path(__file__).with_name('a1_cifar100_worker.py')),
                                    str(root/'launch.json'), family, device], stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    finally:
        log.close()
    handles[family] = dict(process=process, log_path=log_path, offset=0)
    print(f'Launched {family}: PID={process.pid}, device={device}, log={log_path}', flush=True)


def wait_queues(handles):
    last_notice = time.monotonic()
    while True:
        active = []
        for family, handle in handles.items():
            process = handle['process']
            if process is None: continue
            with handle['log_path'].open() as stream:
                stream.seek(handle['offset']); text = stream.read(); handle['offset'] = stream.tell()
            if text: print(text, end='', flush=True)
            code = process.poll()
            if code is None: active.append(family)
            else: handle['returncode'] = code
        if not active: break
        if time.monotonic()-last_notice >= 30:
            print('Waiting for queues:', ', '.join(active), '(see per-run heartbeat.json)', flush=True)
            last_notice = time.monotonic()
        time.sleep(1)
    results = {family: handle.get('returncode') for family, handle in handles.items()}
    print('Queue exit codes:', results, flush=True)
    return results


def interrupt_queues(handles):
    """Explicit stop; last completed epoch remains resumable in each run."""
    import signal
    for handle in handles.values():
        process = handle['process']
        if process is not None and process.poll() is None:
            process.send_signal(signal.SIGINT)
    return wait_queues(handles)


### Metric cùng predictions và diagnostics100 lớp

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile {RUNTIME}/a1_cifar100_report.py
"""One FP32 prediction source, diagnostics, isolated benchmarks, guarded summaries."""
import json
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from a1_cifar100_common import ALL_IDS, MAIN_IDS, file_hash, run_config, check_identity, write_json, Heartbeat, run_log
from a1_cifar100_data import make_loader, load_raw
from a1_cifar100_train import evaluate
from a1_cifar100_worker import build_model, trained_valid


def prediction_metrics(predictions):
    y, p = predictions['y_true'], predictions['y_pred']
    matrix = confusion_matrix(y, p, labels=list(range(100)))
    precision, recall, f1, support = precision_recall_fscore_support(y, p, labels=list(range(100)), zero_division=0)
    return dict(accuracy=float(np.mean(y == p)), macro_f1=float(f1_score(y, p, labels=list(range(100)),
                      average='macro', zero_division=0))), matrix, pd.DataFrame(dict(
                          class_id=range(100), precision=precision, recall=recall, f1=f1, support=support))


def diagnostics(folder, history, predictions, raw_test, names):
    folder = Path(folder)
    metrics, matrix, classes = prediction_metrics(predictions)
    classes['class_name'] = names
    classes.to_csv(folder/'class_metrics.csv', index=False)
    write_json(folder/'confusion_matrix.json', dict(class_names=names, counts=matrix.tolist()))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for key in ('train_loss', 'val_loss'): axes[0].plot(history.epoch+1, history[key], label=key)
    for key in ('train_accuracy', 'val_accuracy'): axes[1].plot(history.epoch+1, history[key], label=key)
    for axis in axes: axis.legend(); axis.grid(alpha=.2); axis.set_xlabel('Epoch')
    fig.tight_layout(); fig.savefig(folder/'curves.png', dpi=150); plt.close(fig)
    fig, axis = plt.subplots(figsize=(12, 11))
    normalized = matrix / np.maximum(matrix.sum(1, keepdims=True), 1)
    im = axis.imshow(normalized, vmin=0, vmax=1, cmap='Blues')
    axis.set(xlabel='Predicted class ID', ylabel='True class ID', title='FP32 normalized confusion (100 classes)')
    fig.colorbar(im, ax=axis); fig.tight_layout(); fig.savefig(folder/'confusion_matrix.png', dpi=150); plt.close(fig)
    offdiag = matrix.copy(); np.fill_diagonal(offdiag, 0)
    pairs = [dict(true=names[i], predicted=names[j], count=int(offdiag[i, j]))
             for i, j in (np.unravel_index(k, matrix.shape) for k in np.argsort(offdiag.ravel())[::-1][:20]) if offdiag[i,j] > 0]
    write_json(folder/'top_confusions.json', pairs)
    wrong = np.flatnonzero(predictions['y_true'] != predictions['y_pred'])
    confidence = predictions['probabilities'].max(1)
    chosen = wrong[np.argsort(confidence[wrong])[::-1][:16]]
    fig, axes = plt.subplots(4, 4, figsize=(13, 11))
    failures = []
    for axis in axes.flat: axis.axis('off')
    for axis, i in zip(axes.flat, chosen):
        index = int(predictions['indices'][i])
        axis.imshow(raw_test.data[index])
        y, p = int(predictions['y_true'][i]), int(predictions['y_pred'][i])
        axis.set_title(f'T: {names[y]}\nP: {names[p]} ({confidence[i]:.2f})', fontsize=8)
        failures.append(dict(index=index, true=y, predicted=p, confidence=float(confidence[i])))
    if not len(chosen): fig.suptitle('No errors in the evaluated samples (not proof of generalization)')
    fig.tight_layout(); fig.savefig(folder/'failures.png', dpi=150); plt.close(fig)
    write_json(folder/'failures.json', failures)
    return metrics


### Benchmark FP32 tuần tự sau khi workers thoát

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_report.py
@torch.no_grad()
def benchmark(model, device, batch_size, smoke=False):
    model.eval().float()
    sample = torch.zeros(batch_size, 3, 224, 224, device=device)
    warmup, repeats = (1, 2) if smoke else (10, 50)
    for _ in range(warmup): model(sample)
    if device.type == 'cuda':
        torch.cuda.synchronize(device); torch.cuda.reset_peak_memory_stats(device)
    start = time.perf_counter()
    for _ in range(repeats): model(sample)
    if device.type == 'cuda': torch.cuda.synchronize(device)
    seconds = time.perf_counter()-start
    return dict(batch_size=batch_size, warmup=warmup, repeats=repeats, precision='fp32', device=str(device),
                device_name=torch.cuda.get_device_name(device) if device.type == 'cuda' else 'cpu',
                latency_ms_per_image=seconds*1000/(batch_size*repeats), images_per_second=batch_size*repeats/seconds,
                peak_allocated_memory_mb=torch.cuda.max_memory_allocated(device)/2**20 if device.type == 'cuda' else 0.)


def evaluate_and_benchmark(config, handles):
    # Enforce actual process liveness, not a caller-supplied boolean.
    if any(handle['process'] is not None and handle['process'].poll() is None for handle in handles.values()):
        raise RuntimeError('Workers still running; benchmark is forbidden')
    root = Path(config['root'])
    manifest = json.loads((root/'manifest.json').read_text())
    train, test = load_raw(config)
    device = torch.device('cpu' if config['cpu'] else 'cuda:0')
    if device.type == 'cuda': torch.cuda.set_device(device)
    torch.set_num_threads(2)
    failures = []
    for experiment in config['experiment_ids'] or ALL_IDS:
        folder = root/experiment
        rc = run_config(config, manifest, experiment)
        if not trained_valid(folder, rc):
            failures.append(experiment); continue
        with Heartbeat(folder/'evaluation_heartbeat.json', config['heartbeat_seconds']) as heartbeat, run_log(folder/'evaluate.log'):
            try:
                write_json(folder/'status.json', dict(state='evaluating', fingerprint=rc['fingerprint']))
                model = build_model(rc, config, device).float()
                try:
                    checkpoint = torch.load(folder/'best.pt', map_location='cpu', weights_only=False)
                    check_identity(checkpoint['config'], rc); model.load_state_dict(checkpoint['model'])
                    history = pd.read_csv(folder/'history.csv')
                    scores, predictions = {}, {}
                    for split in ('train', 'val', 'test'):
                        heartbeat.progress = dict(experiment=experiment, phase='evaluate', split=split)
                        ids = manifest['indices'][split]
                        if config['run_mode'] == 'smoke': ids = ids[:config['batch_size']]
                        raw = test if split == 'test' else train
                        stats, pred = evaluate(model, make_loader(raw, ids, config), device)
                        scores[split] = {**stats, **prediction_metrics(pred)[0]}
                        predictions[split] = pred
                        np.savez_compressed(folder/f'{split}_predictions.npz', **pred)
                    diagnostics(folder, history, predictions['test'], test, manifest['classes'])
                    params = sum(p.numel() for p in model.parameters())
                    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
                    metrics = dict(experiment_id=experiment, fingerprint=rc['fingerprint'], run_mode=config['run_mode'],
                        checkpoint_sha256=file_hash(folder/'best.pt'), epochs_completed=len(history), best_epoch=checkpoint['epoch']+1,
                        precision='fp32', train_precision='fp32' if config['cpu'] else 'amp_fp16',
                        train_accuracy=scores['train']['accuracy'], val_accuracy=scores['val']['accuracy'],
                        test_accuracy=scores['test']['accuracy'], macro_f1=scores['test']['macro_f1'],
                        generalization_gap=scores['train']['accuracy']-scores['val']['accuracy'],
                        evaluated_sizes={k: v['items'] for k, v in scores.items()}, scores=scores,
                        training_seconds=float(history.training_seconds.sum()),
                        parameters=params, trainable_parameters=trainable,
                        peak_training_memory_mb=float(history.peak_training_memory_mb.max()),
                        training_execution=config['execution'])
                    write_json(folder/'metrics.json', metrics)
                    heartbeat.progress = dict(experiment=experiment, phase='benchmark')
                    bench = {str(batch): benchmark(model, device, batch, config['run_mode']=='smoke') for batch in (1, 32)}
                    write_json(folder/'benchmark.json', dict(fingerprint=rc['fingerprint'],
                                checkpoint_sha256=metrics['checkpoint_sha256'], measurements=bench))
                    write_json(folder/'status.json', dict(state='complete', fingerprint=rc['fingerprint']))
                    print(experiment, 'test=', round(metrics['test_accuracy'], 4), flush=True)
                finally:
                    del model
                    if device.type == 'cuda': torch.cuda.empty_cache()
            except Exception as exc:
                failures.append(experiment)
                write_json(folder/'status.json', dict(state='evaluation_failed', fingerprint=rc['fingerprint'], error=str(exc)))
                print(f'{experiment}: evaluation failed: {exc}', flush=True)
    return failures


### Kiểm tra artifact full, bảng chính và augmentation ablation

**Đầu vào:** các định nghĩa Python bên dưới. **Đầu ra:** module worker độc lập; cell đầu ghi mới, các cell sau nối phần tiếp theo. Chạy đúng thứ tự từ kernel sạch.


In [ ]:
%%writefile -a {RUNTIME}/a1_cifar100_report.py
def valid_full(folder, rc, manifest):
    try:
        folder = Path(folder)
        if rc['run_mode'] != 'full' or not trained_valid(folder, rc): return False
        metrics = json.loads((folder/'metrics.json').read_text())
        bench = json.loads((folder/'benchmark.json').read_text())
        status = json.loads((folder/'status.json').read_text())
        for obj in (metrics, bench, status): check_identity(obj, rc)
        if status['state'] != 'complete' or metrics['run_mode'] != 'full': return False
        if metrics['epochs_completed'] != rc['recipe']['epochs'] or metrics['evaluated_sizes'] != manifest['sizes']: return False
        if not metrics['checkpoint_sha256'] == bench['checkpoint_sha256'] == file_hash(folder/'best.pt'): return False
        for key in ('train_accuracy', 'val_accuracy', 'test_accuracy', 'macro_f1'):
            if not np.isfinite(metrics[key]) or not 0 <= metrics[key] <= 1: return False
        for key in ('training_seconds', 'peak_training_memory_mb', 'parameters', 'trainable_parameters'):
            if not np.isfinite(metrics[key]) or metrics[key] < 0: return False
        if not 0 < metrics['trainable_parameters'] <= metrics['parameters']: return False
        for batch in (1, 32):
            row = bench['measurements'][str(batch)]
            if row['precision'] != 'fp32' or row['device'] != 'cuda:0' or row['batch_size'] != batch: return False
            if row['warmup'] != 10 or row['repeats'] != 50: return False
            for key in ('latency_ms_per_image', 'images_per_second', 'peak_allocated_memory_mb'):
                if not np.isfinite(row[key]) or row[key] <= 0: return False
        required = ('curves.png', 'confusion_matrix.json', 'confusion_matrix.png', 'class_metrics.csv',
                    'failures.json', 'failures.png', 'top_confusions.json', 'train_predictions.npz',
                    'val_predictions.npz', 'test_predictions.npz')
        return all((folder/name).is_file() for name in required)
    except (OSError, ValueError, KeyError, TypeError):
        return False


def summarize(config):
    root = Path(config['root'])
    manifest = json.loads((root/'manifest.json').read_text())
    rows, missing = [], []
    for experiment in ALL_IDS:
        rc = run_config(config, manifest, experiment)
        if not valid_full(root/experiment, rc, manifest): missing.append(experiment); continue
        row = json.loads((root/experiment/'metrics.json').read_text())
        bench = json.loads((root/experiment/'benchmark.json').read_text())
        row = {k: v for k, v in row.items() if not isinstance(v, dict)}
        row['latency_ms_per_image_batch32'] = bench['measurements']['32']['latency_ms_per_image']
        rows.append(row)
    frame = pd.DataFrame(rows)
    main = frame[frame.experiment_id.isin(MAIN_IDS)] if len(frame) else frame
    main.to_csv(root/'main_summary.csv', index=False)
    ablation = []
    for family in ('resnet18', 'deit_tiny'):
        ids = (family+'_scratch', family+'_scratch_noaug')
        if not len(frame) or not set(ids) <= set(frame.experiment_id): continue
        a, b = (frame.set_index('experiment_id').loc[i] for i in ids)
        ablation.append(dict(family=family, **{f'delta_{key}':float(a[key]-b[key]) for key in (
            'val_accuracy', 'test_accuracy', 'macro_f1', 'generalization_gap')}))
    ablation_frame = pd.DataFrame(ablation)
    ablation_frame.to_csv(root/'augmentation_ablation.csv', index=False)
    lines = ['# CIFAR-100 measured progress', f'Validated main: {len(main)}/8; ablation pairs: {len(ablation)}/2.',
             'Missing/incomplete: '+', '.join(missing),
             'Training times are concurrent queue costs, not isolated GPU speed.',
             'One seed and a fixed budget do not establish convergence or statistical significance.',
             'Only scratch augmentation effects were tested; test did not select the recipe.']
    if len(main):
        best = main.loc[main.val_accuracy.idxmax()]
        lines.append(f'Highest measured validation accuracy: {best.experiment_id}: {best.val_accuracy:.4f}; corresponding test={best.test_accuracy:.4f}.')
        for family in ('resnet18', 'deit_tiny'):
            group = main[main.experiment_id.str.startswith(family+'_')]
            if len(group):
                winner = group.loc[group.val_accuracy.idxmax()]
                lines.append(f'{family}: best by validation = {winner.experiment_id}; never assume full fine-tune wins.')
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        main.set_index('experiment_id')[['val_accuracy','test_accuracy','macro_f1']].plot.bar(ax=axes[0])
        axes[0].set_ylim(0, 1)
        main.set_index('experiment_id').training_seconds.plot.bar(ax=axes[1]); axes[1].set_title('Concurrent training seconds')
        axes[2].scatter(main.trainable_parameters, main.test_accuracy)
        for _, r in main.iterrows(): axes[2].annotate(r.experiment_id, (r.trainable_parameters,r.test_accuracy), fontsize=6)
        axes[2].set(xlabel='Trainable parameters', ylabel='Test accuracy')
        fig.tight_layout(); fig.savefig(root/'comparison.png', dpi=150); plt.close(fig)
    if len(ablation_frame):
        ablation_frame.set_index('family').plot.bar(figsize=(10, 4))
        plt.axhline(0, color='black', linewidth=.6); plt.ylabel('Augmented minus noaug'); plt.tight_layout()
        plt.savefig(root/'augmentation_delta.png', dpi=150); plt.close()
    (root/'report.md').write_text('\n\n'.join(lines))
    write_json(root/'completion.json', dict(main_completed=len(main), ablation_pairs_completed=len(ablation),
                                          all10_full_complete=not missing, missing=missing))
    return main, ablation_frame, missing


### Freeze study, cache pretrained trên CPU và kiểm tra thiết bị

**Đầu vào:** CONFIG, manifest, EDA; tất cả module đã ghi xong. **Đầu ra:** study fingerprint và launch.json cố định. Parent không giữ model GPU; weights có hash và nguồn. Kiểm tra freeze/output100 lớp chạy trong mỗi worker trước train.


In [ ]:
from a1_cifar100_launch import check_devices
from a1_cifar100_data import prepare
check_devices(CONFIG)
CONFIG, manifest = prepare(CONFIG,manifest,EDA_ROOT)
HANDLES = {}
print('Study root:',CONFIG['root'])
print('8 main:',MAIN_IDS)
print('2 controls:',[item for item in ALL_IDS if item not in MAIN_IDS])


### Queue riêng ResNet-18 → GPU0

**Đầu vào:** study đã freeze. **Đầu ra:** process handle và log; dual trả quyền điều khiển ngay để cell DeiT khởi chạy song song. Không chạy lại launch cell khi process cũ còn sống.


In [ ]:
from a1_cifar100_launch import launch_family
launch_family(CONFIG,'resnet18',HANDLES)


### Queue riêng DeiT-Tiny → GPU1

**Đầu vào:** study và HANDLES. **Đầu ra:** process DeiT độc lập; sequential chủ động chờ ResNet rồi dùng GPU0.


In [ ]:
launch_family(CONFIG,'deit_tiny',HANDLES)


### Chờ cả hai queue; giữ checkpoint khi một run lỗi

**Đầu vào:** actual process handles. **Đầu ra:** exit codes, log và trạng thái. Không hủy worker còn lại khi một queue lỗi. Heartbeat chứng minh process còn phản hồi, không chứng minh GPU đang tiến bước.


In [ ]:
from a1_cifar100_launch import wait_queues
EXIT_CODES = wait_queues(HANDLES)
print('Inspect failed run status/log before resuming:',EXIT_CODES)


### Evaluation và benchmark FP32 trên GPU0, không chồng training

**Đầu vào:** best checkpoints từ các run train hoàn tất. **Đầu ra:** một lượt predictions/split cho accuracy/F1/confusion, diagnostics; benchmark batch1/32, warmup10/repeats50. Smoke dùng budget nhỏ riêng, không nhập bảng full.


In [ ]:
from a1_cifar100_report import evaluate_and_benchmark
failed_evaluations = evaluate_and_benchmark(CONFIG,HANDLES)
print('Incomplete/failed evaluations:',failed_evaluations)


### Bảng chính, hai cặp đối chứng và tiến độ trung thực

**Đầu vào:** artifact được kiểm chứng theo fingerprint/epoch/evaluation/benchmark. **Đầu ra:** CSV, report.md, plots, completion.json. Smoke hoặc thiếu run chỉ hiển thị tiến độ; không tuyên bố đã hoàn thành A1.


In [ ]:
from a1_cifar100_report import summarize
main_results, augmentation_results, missing = summarize(CONFIG)
columns = ['experiment_id','val_accuracy','test_accuracy','macro_f1','training_seconds','trainable_parameters','peak_training_memory_mb','latency_ms_per_image_batch32']
display(main_results[[col for col in columns if col in main_results]])
display(augmentation_results)
ROOT = Path(CONFIG['root'])
for filename in ('comparison.png','augmentation_delta.png'):
    if (ROOT/filename).exists(): display(DisplayImage(filename=str(ROOT/filename)))
display(Markdown((ROOT/'report.md').read_text()))
print('ALL10 FULL COMPLETE' if not missing else f'Chưa hoàn tất: {len(missing)}/10 run thiếu/chưa hợp lệ')


## 7. Diễn giải và tái lập

Đọc bảng để xác định winner theo validation, không chọn theo test và không viết sẵn full tốt nhất. So scratch/pretrained trong từng họ; xem train/val curves để phân biệt underfitting, chưa hội tụ và overfitting. Hai kiến trúc khác kích thước và recipe pretraining; không quy toàn bộ khác biệt cho attention/convolution.

Đọc top_confusions/class_metrics/failure grids để nêu ít nhất3 cặp lớp nhầm thực tế. Đối chứng augmentation chỉ có scratch: delta có thể âm; một seed không đủ kết luận thống kê. Upscale32→224 không khôi phục chi tiết. Exact duplicate audit không bảo đảm không có near-duplicate hoặc overlap với pretraining.

Training time gồm loading/forward/backward/optimizer khi hai queue đồng thời, loại validation/checkpoint I/O; không diễn giải như isolated GPU speed. Benchmark riêng cùng GPU0/FP32/batch/warmup/repeats mới dùng so inference.

**Ngắt chủ động:** gọi `from a1_cifar100_launch import interrupt_queues; interrupt_queues(HANDLES)` trong cell mới. Chỉ resume sau khi process cũ đã thoát; last epoch hoàn tất vẫn được giữ.

**Resume:** giữ nguyên toàn bộ thư mục study; sửa resume_from theo experiment='last', chạy lại từ kernel sạch với cùng recipe. Có thể dùng đường dẫn last.pt tuyệt đối; checkpoint lưu đủ history/best/RNG. Đổi đường dẫn dữ liệu/artifact không đổi study identity nếu nội dung và code giống nhau. Đổi epoch/augmentation/software/code tạo study khác; không ép nạp checkpoint cũ.

**Evaluate-only:** yêu cầu trained.json/best.pt/history tương thích. Có thể chọn experiment_ids để hoàn tất phần thiếu; tổng hợp chỉ đếm đủ10 artifact hợp lệ. Dataset/pretrained tải lỗi gây lỗi thật. Không bật tự động synthetic.

**Trước nộp A1.2:** chạy smoke trên T4×2; thử ngắt/resume; chạy đủ8 main+2 controls full; lưu artifact và notebook đã thực thi; viết nhận xét dựa trên số liệu, PDF và trang A1. Notebook này không tự cập nhật dashboard/PDF.


### 8. Nén output của study hiện tại thành ZIP

**Đầu vào:** `CONFIG['root']` của study đã chạy và HANDLES. **Đầu ra:** ZIP có link tải, gồm EDA, logs, metrics, predictions và cả checkpoint để lưu/resume. Giữ cấu trúc `A1/protocol/fingerprint/mode/` khi giải nén; không đóng gói dataset hoặc pretrained cache. Có thể zip run còn thiếu/lỗi sau khi workers đã dừng; ZIP không có nghĩa đã hoàn tất full. File ZIP nằm ngoài thư mục study để không tự nén chính nó.


In [ ]:
import os
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

if any(handle['process'] is not None and handle['process'].poll() is None
       for handle in HANDLES.values()):
    raise RuntimeError('Chờ hoặc dừng các worker trước khi zip output.')

study_root = Path(CONFIG['root']).resolve()
output_root = Path(CONFIG['output_root']).resolve()
if not study_root.is_dir():
    raise FileNotFoundError(f'Không tìm thấy output: {study_root}')
study_root.relative_to(output_root)  # Bảo đảm archive giữ đúng hierarchy artifact.
zip_path = output_root.parent / f"a1_cifar100_{CONFIG['study_fingerprint']}_{CONFIG['run_mode']}.zip"
temporary_zip = zip_path.with_suffix('.zip.tmp')
files = sorted(path for path in study_root.rglob('*')
               if path.is_file() and not path.is_symlink() and not path.name.endswith('.tmp'))
if not files:
    raise RuntimeError('Output trống, chưa có file để zip.')

try:
    with zipfile.ZipFile(temporary_zip, 'w', compression=zipfile.ZIP_DEFLATED,
                         compresslevel=1, allowZip64=True) as archive:
        for path in files:
            archive.write(path, arcname=path.relative_to(output_root).as_posix())
    temporary_zip.replace(zip_path)
except BaseException:
    temporary_zip.unlink(missing_ok=True)
    raise

print(f'Đã nén {len(files)} file, {zip_path.stat().st_size / 2**20:.1f} MiB: {zip_path}')
display(FileLink(os.path.relpath(zip_path, Path.cwd())))
